### Descomposición estacional + Pronóstico 1 año usando SARIMA

### Importación dataset

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ── 1. CARGAR DATOS 
df = pd.read_csv('Ingresos Alimentadoras.csv', encoding='latin1')

for col in ['Monto Total Efectivo', 'Monto Total Tarjetas']:
    df[col] = df[col].replace(r'[\$,]', '', regex=True)
    df[col] = df[col].replace(r'^\s*-\s*$', '0', regex=True)
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df['Ingreso Total'] = df['Monto Total Efectivo'] + df['Monto Total Tarjetas']
df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True)

ts_full = df.groupby('Fecha')['Ingreso Total'].sum()
ts_full = ts_full.asfreq('D').interpolate()
ts_full = ts_full.resample('W').sum()

# ── FILTROS DE TEMPORALIDAD 
ts            = ts_full[(ts_full.index >= '2023-01-01') & (ts_full.index <= '2025-12-31')]
ts_real_2026  = ts_full[ts_full.index >= '2026-01-01']

print(f"Entrenamiento: {ts.index.min().date()} → {ts.index.max().date()} ({len(ts)} semanas)")
print(f"Real 2026:     {ts_real_2026.index.min().date()} → {ts_real_2026.index.max().date()} ({len(ts_real_2026)} semanas)")

# ── 2. TEST ADF 
print("\n--- Test ADF (estacionariedad) ---")
adf = adfuller(ts)
print(f"  Estadístico: {adf[0]:.4f}")
print(f"  p-value:     {adf[1]:.4f}")
if adf[1] < 0.05:
    print("  → Serie ESTACIONARIA (p < 0.05)")
else:
    print("  → Serie NO estacionaria, se aplicará diferenciación (d=1)")

# ── 3. ACF Y PACF ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(ts.diff().dropna(),  lags=20, ax=axes[0], title='ACF – Serie diferenciada')
plot_pacf(ts.diff().dropna(), lags=20, ax=axes[1], title='PACF – Serie diferenciada')
plt.tight_layout()
plt.savefig('acf_pacf.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n Guardado: acf_pacf.png")

# ── 4. AJUSTAR SARIMA(0,1,1)(0,1,1)[52] 
print("\n--- Ajustando SARIMA(0,1,1)(0,1,1)[52] ---")
print("    (esto puede tardar unos minutos...)")

modelo = SARIMAX(ts,
                 order=(0, 1, 1),
                 seasonal_order=(0, 1, 1, 52),
                 enforce_stationarity=False,
                 enforce_invertibility=False)

resultado = modelo.fit(disp=False)
print(resultado.summary())

# ── 5. DIAGNÓSTICOS 
residuales = resultado.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Residuales en el tiempo
axes[0, 0].plot(residuales.index, residuales, color='steelblue', lw=1)
axes[0, 0].axhline(0, color='red', linestyle='--', lw=0.8)
axes[0, 0].set_title('Residuales en el tiempo')
axes[0, 0].grid(True, alpha=0.3)

# Histograma + KDE
residuales.plot(kind='hist', bins=20, density=True, ax=axes[0, 1],
                color='steelblue', alpha=0.6)
residuales.plot(kind='kde', ax=axes[0, 1], color='tomato', lw=2)
axes[0, 1].set_title('Distribución de residuales')
axes[0, 1].grid(True, alpha=0.3)

# ACF de residuales
plot_acf(residuales.dropna(), lags=15, ax=axes[1, 0],
         title='ACF – Residuales')

# PACF de residuales
plot_pacf(residuales.dropna(), lags=15, ax=axes[1, 1],
          title='PACF – Residuales')

plt.suptitle('Diagnósticos del Modelo SARIMA(0,1,1)(0,1,1)[52]', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('diagnosticos_sarima.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n Guardado: diagnosticos_sarima.png")
# ── 6. PRONÓSTICO: desde inicio 2026 hasta fin 2027 ───────────
ultima_semana_train = ts.index[-1]
fin_proyeccion      = pd.Timestamp('2026-12-31')
steps = int((fin_proyeccion - ultima_semana_train).days / 7) + 1

pronostico = resultado.get_forecast(steps=steps)
media      = pronostico.predicted_mean
ic         = pronostico.conf_int(alpha= 0.20)

# Separar zona con dato real vs. futuro puro
media_2026   = media[media.index <= ts_real_2026.index.max()]
media_futuro = media[media.index >  ts_real_2026.index.max()]
ic_2026      = ic[ic.index <= ts_real_2026.index.max()]
ic_futuro    = ic[ic.index >  ts_real_2026.index.max()]

# ── 7. GRÁFICAS
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(ts.index, ts / 1e6,
        label='Histórico 2023–2025', color='steelblue', lw=1.5)

ax.plot(ts_real_2026.index, ts_real_2026 / 1e6,
        label='Real 2026', color='seagreen', lw=2)

ax.plot(media_2026.index, media_2026 / 1e6,
        label='Proyección 2026 (vs real)', color='tomato', lw=1.8, linestyle='--')
ax.fill_between(ic_2026.index,
                ic_2026.iloc[:, 0] / 1e6,
                ic_2026.iloc[:, 1] / 1e6,
                alpha=0.15, color='orange')

ax.plot(media_futuro.index, media_futuro / 1e6,
        label='Proyección 2027', color='tomato', lw=2)
ax.fill_between(ic_futuro.index,
                ic_futuro.iloc[:, 0] / 1e6,
                ic_futuro.iloc[:, 1] / 1e6,
                alpha=0.25, color='orange', label='IC 95%')

ax.axvline(x=ts_real_2026.index.max(), color='gray', linestyle=':', lw=1.2,
           label=f'Último dato real ({ts_real_2026.index.max().date()})')

ax.set_title('Pronóstico de Ingresos – Alimentadoras 2026–2027', fontsize=21, fontweight='bold')
ax.set_ylabel('Millones MXN')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pronostico_sarima.png', dpi=150, bbox_inches='tight')
plt.close()
print(" Guardado: pronostico_sarima.png")

# ── 8. RESUMEN NUMÉRICO ───────────────────────────────────────
print("\n--- Proyección mensual 2026–2027 ---")
df_fc = pd.DataFrame({'Semana': media.index, 'Ingreso': media.values})
df_fc['Mes'] = df_fc['Semana'].dt.to_period('M')
resumen = df_fc.groupby('Mes')['Ingreso'].sum()

for mes, valor in resumen.items():
    print(f"  {mes}  →  ${valor:>12,.0f} MXN")

print(f"\n  Total proyectado 2026–2027: ${resumen.sum():>12,.0f} MXN")
print(f"  AIC del modelo:             {resultado.aic:>12.2f}")
print(f"  BIC del modelo:             {resultado.bic:>12.2f}")

Entrenamiento: 2023-01-01 → 2025-12-28 (157 semanas)
Real 2026:     2026-01-04 → 2026-02-15 (7 semanas)

--- Test ADF (estacionariedad) ---
  Estadístico: -5.9704
  p-value:     0.0000
  → Serie ESTACIONARIA (p < 0.05)

✅ Guardado: acf_pacf.png

--- Ajustando SARIMA(0,1,1)(0,1,1)[52] ---
    (esto puede tardar unos minutos...)
                                     SARIMAX Results                                      
Dep. Variable:                      Ingreso Total   No. Observations:                  157
Model:             SARIMAX(0, 1, 1)x(0, 1, 1, 52)   Log Likelihood                -658.077
Date:                            Wed, 18 Mar 2026   AIC                           1322.154
Time:                                    19:36:53   BIC                           1327.890
Sample:                                01-01-2023   HQIC                          1324.338
                                     - 12-28-2025                                         
Covariance Type:                  